# Digit Recognizer

- Load the libraries
- Load the Data
- Normalize the Data
- Build the architecture
- Split the data

In [29]:
import torch
from matplotlib import pyplot as plt
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn import CrossEntropyLoss

In [30]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [31]:
from tensorflow.keras.datasets import mnist
(x_train, y_train), (x_test, y_test) = mnist.load_data()

In [42]:
x_train = x_train / 255.0
x_test = x_test / 255.0

In [43]:
x_train_tensor = torch.tensor(x_train, dtype=torch.float32, device=device)
x_test_tensor = torch.tensor(x_test, dtype=torch.float32, device=device)

y_train_tensor = torch.tensor(y_train, dtype=torch.int64, device=device)
y_test_tensor = torch.tensor(y_test, dtype=torch.int64, device=device)

In [44]:
class CustomDataset(Dataset):
  def __init__(self, x_features, y_target):
    self.features = x_features
    self.target = y_target

  def __len__(self):
    return len(self.features)

  def __getitem__(self, index):
    return self.features[index], self.target[index]

In [45]:
train_dataset = CustomDataset(x_train_tensor, y_train_tensor)
test_dataset = CustomDataset(x_test_tensor, y_test_tensor)

In [46]:
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [47]:
class DigitRecognizerNeuralNetwork(nn.Module):
  def __init__(self, ):
    super().__init__()
    self.network = nn.Sequential(
        nn.Linear(in_features=784, out_features=512, bias=True),
        nn.ReLU(),
        nn.Linear(in_features=512, out_features=256, bias=True),
        nn.ReLU(),
        nn.Linear(in_features=256, out_features=128, bias=True),
        nn.ReLU(),
        nn.Linear(in_features=128, out_features=64, bias=True),
        nn.ReLU(),
        nn.Linear(in_features=64, out_features=32, bias=True),
        nn.ReLU(),
        nn.Linear(in_features=32, out_features=18, bias=True),
        nn.ReLU(),
        nn.Linear(in_features=18, out_features=10, bias=True)
    )

  def forward(self, x):
    x = x.flatten(start_dim=1)
    x_out = self.network(x)
    return x_out


In [51]:
model = DigitRecognizerNeuralNetwork().to(device=device)
criterion = CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-003)

In [52]:
from datetime import datetime

In [54]:
model.train()
now = datetime.now()
for epoch in range(25):
  train_loss = 0
  for x_batch, y_batch in train_dataloader:
    x_batch = x_batch.to(device)
    y_batch = y_batch.to(device)
    y_pred = model(x_batch)
    loss = criterion(y_pred, y_batch)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    train_loss += loss.item()

  print(f"Epoch {epoch + 1} | Loss {train_loss/len(train_dataloader)} | Time {datetime.now() - now}")



Epoch 1 | Loss 0.41017036824226377 | Time 0:00:05.374583
Epoch 2 | Loss 0.24301504941383997 | Time 0:00:10.453981
Epoch 3 | Loss 0.18223629828890164 | Time 0:00:15.964774
Epoch 4 | Loss 0.15038468927194676 | Time 0:00:21.057276
Epoch 5 | Loss 0.12617593253316978 | Time 0:00:26.637944
Epoch 6 | Loss 0.11236152205945303 | Time 0:00:31.748413
Epoch 7 | Loss 0.09999315860929589 | Time 0:00:37.130807
Epoch 8 | Loss 0.08846171984846393 | Time 0:00:42.413843
Epoch 9 | Loss 0.08145482516624034 | Time 0:00:47.910616
Epoch 10 | Loss 0.07304807341579969 | Time 0:00:53.441519
Epoch 11 | Loss 0.06673090128246695 | Time 0:00:58.537241
Epoch 12 | Loss 0.06185328314799505 | Time 0:01:04.114250
Epoch 13 | Loss 0.057912171199638394 | Time 0:01:09.265156
Epoch 14 | Loss 0.05204716861369088 | Time 0:01:14.820188
Epoch 15 | Loss 0.049743809884848694 | Time 0:01:19.950598
Epoch 16 | Loss 0.0460259402621227 | Time 0:01:25.166981
Epoch 17 | Loss 0.042583965298649856 | Time 0:01:30.687859
Epoch 18 | Loss 0.040

In [61]:
model.eval()
now = datetime.now()

correct = 0
total = 0

with torch.no_grad():
  test_loss = 0
  for x_batch, y_batch in test_dataloader:
    x_batch = x_batch.to(device)
    y_batch = y_batch.to(device)

    pred = model(x_batch)
    something, predicted = torch.max(pred, 1)
    #print(f"Something {something}, predicted {predicted}")

    correct += (predicted == y_batch).sum().item()
    total += y_batch.size(0)
print(f"Accuracy {100 * correct / total:.2f}")




Accuracy 97.46


In [62]:
correct = total = 0

with torch.no_grad():
  for x_batch, y_batch in train_dataloader:
    x_batch = x_batch.to(device)
    y_batch = y_batch.to(device)

    output = model(x_batch)

    _, predicted = torch.max(output, 1)

    correct += (predicted == y_batch).sum().item()
    total += y_batch.size(0)

print(f"Accuracy {100 * correct / total:.2f}")

Accuracy 99.11
